# Vision Privacy & Identity Lab – Module 3
## Final Image Generation

This notebook loads the trained LoRA weights and generates new images using
Stable Diffusion with your unique activation token.

In [ ]:
# ── 1. Install inference stack ────────────────────────────────────────────────
!pip install -q torch torchvision diffusers transformers accelerate xformers Pillow

In [ ]:
# ── 2. Colab + path setup ─────────────────────────────────────────────────────
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/IA-'):
        !git clone -q https://github.com/ap-xlr8/IA- /content/IA-
    sys.path.insert(0, '/content/IA-')
else:
    sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

In [ ]:
# ── 3. Configuration ──────────────────────────────────────────────────────────
BASE_MODEL     = 'runwayml/stable-diffusion-v1-5'
LORA_WEIGHTS   = '/content/drive/MyDrive/vp_lab/model' if IN_COLAB else './model'
OUTPUT_IMAGES  = '/content/drive/MyDrive/vp_lab/generated' if IN_COLAB else './data/generated'

# Must match the token used in training (Module 2)
# Set via environment variable or hard-code here:
import os
ACTIVATION_TOKEN = os.environ.get('VLAB_ACTIVATION_TOKEN', 'vplid_REPLACE_ME')
CLASE = 'person'

DEVICE         = 'cuda'   # 'cuda' or 'cpu'
NUM_IMAGES     = 4
NUM_STEPS      = 30
GUIDANCE_SCALE = 7.5
IMAGE_SIZE     = 512

# Prompts to generate (token + class word will be prepended automatically)
PROMPTS = [
    'smiling, natural lighting, high quality portrait',
    'side profile, outdoor background, photorealistic',
    'professional headshot, studio lighting',
    'candid shot, bokeh background',
]

print(f'Token : {ACTIVATION_TOKEN}')
print(f'Model : {LORA_WEIGHTS}')

In [ ]:
# ── 4. Load pipeline ──────────────────────────────────────────────────────────
import torch
from diffusers import StableDiffusionPipeline
from pathlib import Path

pipe = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
)

# Load LoRA weights if available
lora_path = Path(LORA_WEIGHTS)
if lora_path.exists():
    pipe.load_lora_weights(str(lora_path))
    print(f'LoRA weights loaded from {lora_path}')
else:
    print(f'⚠️  LoRA path not found ({lora_path}). Generating with base model only.')

# Memory optimisations
pipe.enable_xformers_memory_efficient_attention()
pipe = pipe.to(DEVICE)
print('Pipeline ready.')

In [ ]:
# ── 5. Generate images ────────────────────────────────────────────────────────
import math
from PIL import Image
import matplotlib.pyplot as plt

Path(OUTPUT_IMAGES).mkdir(parents=True, exist_ok=True)
all_images = []

for i, prompt_suffix in enumerate(PROMPTS[:NUM_IMAGES]):
    full_prompt = f'{ACTIVATION_TOKEN} {CLASE}, {prompt_suffix}'
    print(f'Generating [{i+1}/{NUM_IMAGES}]: {full_prompt}')

    with torch.autocast(DEVICE):
        result = pipe(
            full_prompt,
            num_inference_steps=NUM_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            height=IMAGE_SIZE,
            width=IMAGE_SIZE,
        )
    img = result.images[0]
    out_path = Path(OUTPUT_IMAGES) / f'gen_{i:04d}.png'
    img.save(out_path)
    all_images.append(img)
    print(f'  Saved → {out_path}')

print(f'\n✅  {len(all_images)} images saved to {OUTPUT_IMAGES}')

In [ ]:
# ── 6. Display results ────────────────────────────────────────────────────────
cols = min(4, len(all_images))
rows = math.ceil(len(all_images) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))

for ax, img in zip(axes.flat if hasattr(axes, 'flat') else [axes], all_images):
    ax.imshow(img)
    ax.axis('off')

# Hide unused axes
if hasattr(axes, 'flat'):
    for ax in list(axes.flat)[len(all_images):]:
        ax.set_visible(False)

plt.tight_layout()
plt.show()